In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import kagglehub
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np


# Print the dataset path so we can confirm where the files were downloaded
print("Path to dataset files:", path)

# Build expected train/test directories (common Kaggle dataset structure)
train_dir = os.path.join(path, "PlantVillage" ,"train")  # Training folder path
test_dir = os.path.join(path,"PlantVillage" ,"test")  # Testing folder path

# Define normalization stats (ImageNet stats are a standard default for RGB models)
mean = [0.485, 0.456, 0.406]  # Per-channel mean
std = [0.229, 0.224, 0.225]  # Per-channel std

# Create the training transform pipeline:
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.RandomRotation(15),  # Randomly rotate images by up to ±15 degrees
    transforms.ToTensor(),  # Convert PIL image to torch tensor in [0,1]
    transforms.Normalize(mean=mean, std=std)  # Normalize tensor channels
])

# Create the testing transform pipeline:
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.ToTensor(),  # Convert PIL image to torch tensor
    transforms.Normalize(mean=mean, std=std)  # Normalize tensor channels
])

# Create datasets using ImageFolder:
# ImageFolder automatically assigns labels based on subfolder names
train_dataset = ImageFolder(root=train_dir, transform=train_transform)  # Training dataset
test_dataset = ImageFolder(root=test_dir, transform=test_transform)  # Testing dataset

# Create DataLoaders:

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # Training DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)  # Testing DataLoader

# Print dataset sizes and class mapping to verify correct loading
print(f"Training samples: {len(train_dataset)}")  # Number of training images
print(f"Testing samples: {len(test_dataset)}")  # Number of testing images
print(f"Class to index mapping: {train_dataset.class_to_idx}")  # Mapping from class name to numeric label

# Fetch one batch to confirm shapes and label ranges
images, labels = next(iter(train_loader))
print(f"Batch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")

# Helper function to denormalize an image tensor for visualization
def denormalize(img_tensor, mean_vals, std_vals):
    # Convert mean/std to numpy arrays for broadcasting
    mean_arr = np.array(mean_vals)  # Shape: (3,)
    std_arr = np.array(std_vals)  # Shape: (3,)

    # Move tensor to CPU, convert to numpy, and change from CHW to HWC
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)  # Shape: (32, 32, 3)

    # Reverse normalization: x = x * std + mean
    img = img * std_arr + mean_arr  # Denormalize pixel values

    # Clip to valid display range
    img = np.clip(img, 0, 1)  # Ensure values are within [0,1]

    return img  # Return display-ready image

# Display a few sample images with their labels
num_samples_to_show = 6  # Number of images to visualize
fig, axes = plt.subplots(1, num_samples_to_show, figsize=(3 * num_samples_to_show, 3))


idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}  # Reverse mapping

for i in range(num_samples_to_show):
    # Take the i-th image from the current batch
    img_tensor = images[i]  # Tensor image
    label_idx = labels[i].item()  # Numeric label

    # Denormalize for correct visualization
    img_vis = denormalize(img_tensor, mean, std)  # Convert to displayable RGB image

    # Plot the image
    axes[i].imshow(img_vis)  # Show image
    axes[i].set_title(idx_to_class[label_idx])  # Set title to class name
    axes[i].axis("off")  # Hide axes for cleaner visualization

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Write your code here
import torch.nn as nn

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()  # Initialize base nn.Module

        # Convolutional block 1: 3 -> 32 channels
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn1 = nn.BatchNorm2d(32)  # BatchNorm after conv
        self.relu1 = nn.ReLU()  # Non-linearity
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # Downsample: 32x32 -> 16x16

        # Convolutional block 2: 32 -> 64 channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn2 = nn.BatchNorm2d(64)  # BatchNorm after conv
        self.relu2 = nn.ReLU()  # Non-linearity
        self.pool2 = nn.MaxPool2d(2, 2)  # Downsample: 16x16 -> 8x8

        # Convolutional block 3: 64 -> 128 channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn3 = nn.BatchNorm2d(128)  # BatchNorm after conv
        self.relu3 = nn.ReLU()  # Non-linearity
        self.pool3 = nn.MaxPool2d(2, 2)  # Downsample: 8x8 -> 4x4

        # Convolutional block 4: 128 -> 256 channels
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn4 = nn.BatchNorm2d(256)  # BatchNorm after conv
        self.relu4 = nn.ReLU()  # Non-linearity
        self.pool4 = nn.MaxPool2d(2, 2)  # Downsample: 4x4 -> 2x2

        # Convolutional block 5: 256 -> 256 channels
        self.conv5 = nn.Conv2d(256, 256, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn5 = nn.BatchNorm2d(256)  # BatchNorm after conv
        self.relu5 = nn.ReLU()  # Non-linearity
        self.pool5 = nn.MaxPool2d(2, 2)  # Downsample: 2x2 -> 1x1

        # Classifier head: flatten 256x1x1 -> 256, then map to num_classes
        self.classifier = nn.Sequential(
            nn.Flatten(),  # Flatten to (B, 256)
            nn.Linear(256, 128),  # Hidden layer
            nn.ReLU(),  # Non-linearity
            nn.Linear(128, num_classes)  # Output logits for 3 classes
        )

    def forward(self, x):
        # Pass through conv blocks sequentially
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))  # Block 1
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))  # Block 2
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))  # Block 3
        x = self.pool4(self.relu4(self.bn4(self.conv4(x))))  # Block 4
        x = self.pool5(self.relu5(self.bn5(self.conv5(x))))  # Block 5

        # Pass through classifier to get logits
        x = self.classifier(x)  # Shape: (B, num_classes)
        return x  # Return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select compute device

# Instantiate the model and move it to the device
model = PotatoCNN(num_classes=3).to(device)  # Create model for 3 classes

# Print model summary (architecture)
print(model)


In [ ]:
# Write your code here
import torch.optim as optim
from tqdm import tqdm


# Define training loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0  # Accumulate loss
    correct = 0  # Count correct predictions
    total = 0  # Count total samples

    for images, labels in tqdm(dataloader):  # Iterate over batches with progress bar
        images, labels = images.to(device), labels.to(device)  # Move batch to device

        outputs = model(images)  # Forward pass -> logits (B, 3)
        loss = criterion(outputs, labels)  # Compute cross-entropy loss

        optimizer.zero_grad()  # Clear old gradients
        loss.backward()  # Backpropagate
        optimizer.step()  # Update weights

        total_loss += loss.item()  # Add batch loss

        # Compute batch accuracy
        predictions = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
        predictions = torch.argmax(predictions, dim=1)  # Pick most likely class
        correct += (predictions == labels).sum().item()  # Count correct
        total += labels.size(0)  # Count samples

    avg_loss = total_loss / len(dataloader)  # Average loss over epoch
    accuracy = 100 * correct / total  # Accuracy percentage
    return avg_loss, accuracy  # Return metrics

# Define validation loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0  # Accumulate loss
    correct = 0  # Count correct predictions
    total = 0  # Count total samples

    with torch.no_grad():  # Disable gradient computation for validation
        for images, labels in dataloader:  # Iterate over validation batches
            images, labels = images.to(device), labels.to(device)  # Move to device

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()  # Accumulate loss

            # Compute accuracy
            predictions = torch.softmax(outputs, dim=1)  # Probabilities
            predictions = torch.argmax(predictions, dim=1)  # Predicted class
            correct += (predictions == labels).sum().item()  # Correct count
            total += labels.size(0)  # Total count

    avg_loss = total_loss / len(dataloader)  # Average loss
    accuracy = 100 * correct / total  # Accuracy percentage
    return avg_loss, accuracy  # Return metrics



In [ ]:
# Write your code here
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class classification loss
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Adam optimizer with small LR

# Set number of epochs
num_epochs = 10  # Train for 10 epochs
# Lists to store metrics for plotting
train_losses = []  # Track training loss
val_losses = []  # Track validation loss
train_accuracies = []  # Track training accuracy
val_accuracies = []  # Track validation accuracy

# Training process
for epoch in range(num_epochs):
    # Train for one epoch
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate after epoch
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # Print epoch summary
    print(
        f"Epoch {epoch+1}/{num_epochs}: "
        f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
        f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%"
    )

# Plot loss and accuracy curves (same plotting style as q1_solution_(3).py)
plt.figure(figsize=(12, 5))  # Create a wide figure

# Loss subplot
plt.subplot(1, 2, 1)  # 1 row, 2 cols, first plot
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss", marker='o')  # Train loss curve
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss", marker='o')  # Val loss curve
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Accuracy subplot
plt.subplot(1, 2, 2)  # 1 row, 2 cols, second plot
plt.plot(range(1, num_epochs + 1), train_accuracies, label="Train Accuracy", marker='o')  # Train acc curve
plt.plot(range(1, num_epochs + 1), val_accuracies, label="Validation Accuracy", marker='o')  # Val acc curve
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:

class PotatoCNNResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNNResidual, self).__init__()  # Initialize base nn.Module

        # Convolutional block 1: 3 -> 32 channels
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn1 = nn.BatchNorm2d(32)  # BatchNorm
        self.relu1 = nn.ReLU()  # ReLU
        self.pool1 = nn.MaxPool2d(2, 2)  # 32x32 -> 16x16

        # Convolutional block 2: 32 -> 64 channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn2 = nn.BatchNorm2d(64)  # BatchNorm
        self.relu2 = nn.ReLU()  # ReLU
        self.pool2 = nn.MaxPool2d(2, 2)  # 16x16 -> 8x8

        # Convolutional block 3: 64 -> 128 channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn3 = nn.BatchNorm2d(128)  # BatchNorm
        self.relu3 = nn.ReLU()  # ReLU
        self.pool3 = nn.MaxPool2d(2, 2)  # 8x8 -> 4x4

        # Convolutional block 4: 128 -> 256 channels
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn4 = nn.BatchNorm2d(256)  # BatchNorm
        self.relu4 = nn.ReLU()  # ReLU
        self.pool4 = nn.MaxPool2d(2, 2)  # 4x4 -> 2x2

        # Convolutional block 5: 256 -> 256 channels
        self.conv5 = nn.Conv2d(256, 256, kernel_size=3, padding=1)  # Preserve spatial size
        self.bn5 = nn.BatchNorm2d(256)  # BatchNorm
        self.relu5 = nn.ReLU()  # ReLU
        self.pool5 = nn.MaxPool2d(2, 2)  # 2x2 -> 1x1

        # Skip connection adapters:
        #Downsample spatialy from 8x8 (after pool2) to 2x2 (to match after pool4 input/output stage)
        #Project channels from 64 -> 256 so we can add to convl4 output
        self.skip_pool = nn.MaxPool2d(kernel_size=4, stride=4)  # 8x8 -> 2x2
        self.skip_proj = nn.Conv2d(64, 256, kernel_size=1)  # Channel projection 64 -> 256
        self.skip_bn = nn.BatchNorm2d(256)  # BatchNorm for the projected skip

        # Classifier head: flatten 256x1x1 -> 256, then map to num_classes
        self.classifier = nn.Sequential(
            nn.Flatten(),  # Flatten to (B, 256)
            nn.Linear(256, 128),  # Hidden layer
            nn.ReLU(),  # Non-linearity
            nn.Linear(128, num_classes)  # Output logits
        )

    def forward(self, x):
        # Block 1
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))  # (B,32,16,16)

        # Block 2
        x2 = self.pool2(self.relu2(self.bn2(self.conv2(x))))  # (B,64,8,8)
        # Save x2 for skip connection (this is the "from conv2" feature map)
        skip = x2  # (B,64,8,8)

        # Block 3
        x3 = self.pool3(self.relu3(self.bn3(self.conv3(x2))))  # (B,128,4,4)

        # Block 4 (compute conv4 features first)
        x4 = self.pool4(self.relu4(self.bn4(self.conv4(x3))))  # (B,256,2,2)

        # Prepare skip to match x4 shape:
        # - Downsample 8x8 -> 2x2
        # - Project 64 -> 256 channels
        skip = self.skip_pool(skip)  # (B,64,2,2)
        skip = self.skip_bn(self.skip_proj(skip))  # (B,256,2,2)

        # Residual summation (skip connection from conv2 -> conv4)
        x4 = x4 + skip  # (B,256,2,2)

        # Block 5
        x5 = self.pool5(self.relu5(self.bn5(self.conv5(x4))))  # (B,256,1,1)

        # Classifier
        out = self.classifier(x5)  # (B,num_classes)
        return out  # Return logits

# Instantiate the residual model and move it to the device
res_model = PotatoCNNResidual(num_classes=3).to(device)  # Create residual model

# Print residual model architecture for verification
print(res_model)

# Define loss function and optimizer for residual model
criterion = nn.CrossEntropyLoss()  # Cross-entropy loss
optimizer = optim.Adam(res_model.parameters(), lr=0.0001)  # Adam optimizer

# Set number of epochs for residual training
num_epochs = 10  # Train for 10 epochs

# Lists to store residual model metrics
res_train_losses = []  # Training loss history
res_val_losses = []  # Validation loss history
res_train_accuracies = []  # Training accuracy history
res_val_accuracies = []  # Validation accuracy history

# Train the residual model
for epoch in range(num_epochs):
    # Train for one epoch
    train_loss, train_accuracy = train_one_epoch(res_model, train_loader, criterion, optimizer, device)

    # Validate after epoch
    val_loss, val_accuracy = validate(res_model, test_loader, criterion, device)

    # Store metrics
    res_train_losses.append(train_loss)
    res_val_losses.append(val_loss)
    res_train_accuracies.append(train_accuracy)
    res_val_accuracies.append(val_accuracy)

    # Print epoch summary
    print(
        f"[Residual] Epoch {epoch+1}/{num_epochs}: "
        f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
        f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%"
    )

# Plot residual model loss and accuracy curves (same plotting style as earlier)
plt.figure(figsize=(12, 5))  # Create a wide figure

# Loss subplot
plt.subplot(1, 2, 1)  # 1 row, 2 cols, first plot
plt.plot(range(1, num_epochs + 1), res_train_losses, label="Train Loss (Residual)", marker='o')  # Train loss
plt.plot(range(1, num_epochs + 1), res_val_losses, label="Validation Loss (Residual)", marker='o')  # Val loss
plt.xlabel("Epochs")  # X label
plt.ylabel("Loss")  # Y label
plt.title("Residual Model Loss Curve")  # Title
plt.legend()  # Legend

# Accuracy subplot
plt.subplot(1, 2, 2)  # 1 row, 2 cols, second plot
plt.plot(range(1, num_epochs + 1), res_train_accuracies, label="Train Accuracy (Residual)", marker='o')  # Train acc
plt.plot(range(1, num_epochs + 1), res_val_accuracies, label="Validation Accuracy (Residual)", marker='o')  # Val acc
plt.xlabel("Epochs")  # X label
plt.ylabel("Accuracy (%)")  # Y label
plt.title("Residual Model Accuracy Curve")  # Title
plt.legend()  # Legend

plt.show()  # Render plots